In [ ]:
# We are going to create and train a self_supervised PatchTST model, and then fine-tune it using supervised learning.
# The model is based on the paper "Self-Supervised Learning of Patch Transformers for Time Series Classification" by Xu et al. (2022).

# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader, Subset

import random
import os
import numpy as np
import pandas as pd
import math
import sys
from collections import Counter
from itertools import chain
from typing import List
import textwrap

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from google.colab import drive

random.seed(0)
torch.manual_seed(0)

In [ ]:
drive.mount('/content/gdrive')

base_dir = "/content/gdrive/MyDrive/CS4782-Final/"
sys.path.append(base_dir)

Mounted at /content/gdrive


In [ ]:
%load_ext autoreload
%autoreload 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

Device set to cpu


In [ ]:
from PatchTST_self_supervised import PatchTSTSelfSupervised, self_supervised_loss
from PatchTST import PatchTST
from utils import train, val, train_self_supervised, val_self_supervised

In [ ]:
from data_loader import TimeSeriesDataset, SelfSupervisedTimeSeriesDataset
from sklearn.preprocessing import StandardScaler

input_length    = 512   # e.g. past 336 steps
forecast_horizon= 96    # e.g. next 96 steps
batch_size      = 32

df   = pd.read_csv(base_dir + "data/ETTh1.csv", parse_dates=["date"])
vars = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
data = df[vars].values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Build datasets & loaders
selfsup_train_ds = SelfSupervisedTimeSeriesDataset(train_data, input_length)
selfsup_val_ds   = SelfSupervisedTimeSeriesDataset(val_data, input_length)

selfsup_train_loader = DataLoader(selfsup_train_ds, batch_size=32, shuffle=True, drop_last=True)
selfsup_val_loader   = DataLoader(selfsup_val_ds, batch_size=32, shuffle=False)


In [ ]:
criterion = nn.MSELoss()
model = PatchTSTSelfSupervised(
    input_length=input_length,
    patch_len=12,
    stride=12,
    n_heads=4,
    d_model=16
).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
print('here')
train_loss_arr, val_loss_arr = train_self_supervised(
    model, selfsup_train_loader, selfsup_val_loader,
    self_supervised_loss=self_supervised_loss, epochs=20,
    optimizer=optimizer, device=device
)

# Get the state_dict of the model's encoder
transformer_state_dict = model.transformer.state_dict()

In [ ]:
df   = pd.read_csv(base_dir + "data/ETTh1.csv", parse_dates=["date"])
vars = ['HUFL','HULL','MUFL','MULL','LUFL','LULL','OT']
data = df[vars].values

# 2) Determine time‐based split point
n_train_time = int(0.8 * len(data))

# 3) Slice train & val (include overlap of input_length for first val window)
train_raw = data[:n_train_time]
val_raw   = data[n_train_time - input_length :]

# 4) Fit scaler on TRAIN ONLY, then transform both
scaler     = StandardScaler().fit(train_raw)
train_data = scaler.transform(train_raw)
val_data   = scaler.transform(val_raw)

# 5) Array‐based dataset
class ArrayTimeSeriesDataset(Dataset):
    def __init__(self, data: np.ndarray, input_length: int, horizon: int):
        """
        data: 2D array [T, n_vars] already scaled
        """
        self.data = data
        self.L, self.h = input_length, horizon
        self.n_samples = len(data) - input_length - horizon + 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.L]                       # [L, n_vars]
        y = self.data[idx + self.L : idx + self.L + self.h]     # [h, n_vars]
        # transpose to [n_vars, seq_len]
        return (
            torch.from_numpy(x.T).float(),
            torch.from_numpy(y.T).float()
        )

# 6) Build datasets & loaders
train_ds = ArrayTimeSeriesDataset(train_data, input_length, forecast_horizon)
val_ds   = ArrayTimeSeriesDataset(val_data,   input_length, forecast_horizon)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=False)

In [ ]:
# Create a new PatchTST model for supervised training
supervised_model = PatchTST(
    input_length=input_length,
    patch_len=16,
    stride=16,
    n_heads=4,
    d_model=16,
    forecast_horizon=forecast_horizon
).to(device)
optimizer = optim.Adam(supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
# Load the encoder state_dict into the new model
supervised_model.transformer.load_state_dict(transformer_state_dict)

supervised_model.transformer.requires_grad = True

fine_tune_train_loss_arr, _, fine_tune_val_loss_arr, _ = train(
    supervised_model, train_loader, val_loader,
    criterion=criterion, epochs=10,
    optimizer=optimizer, device=device
)

In [ ]:
# Create a new PatchTST model for supervised training
normal_supervised_model = PatchTST(
    input_length=input_length,
    patch_len=16,
    stride=16,
    forecast_horizon=forecast_horizon
).to(device)
optimizer = optim.Adam(normal_supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)

normal_sup_train_loss_arr, _, normal_sup_val_loss_arr, _ = train(
    normal_supervised_model, train_loader, val_loader,
    criterion=criterion, epochs=10,
    optimizer=optimizer, device=device
)

In [ ]:
forecast_horizon_windows = [24, 48, 96, 192, 336, 720]

end_val_loss = []

for horizon in forecast_horizon_windows:
    print(f"Horizon: {horizon}")
    print("-" * 20)
    model = PatchTSTSelfSupervised(
        input_length=input_length,
        patch_len=12,
        stride=12,
        n_heads=4,
        d_model=16
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
    train_loss_arr, val_loss_arr = train_self_supervised(
        model, selfsup_train_loader, selfsup_val_loader,
        self_supervised_loss=self_supervised_loss, epochs=20,
        optimizer=optimizer, device=device
    )

    # Get the state_dict of the model's encoder
    transformer_state_dict = model.transformer.state_dict()

    supervised_model = PatchTST(
        input_length=input_length,
        patch_len=16,
        stride=16,
        n_heads=4,
        d_model=16,
        forecast_horizon=horizon
    ).to(device)
    optimizer = optim.Adam(supervised_model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
    # Load the encoder state_dict into the new model
    supervised_model.transformer.load_state_dict(transformer_state_dict)

    supervised_model.transformer.requires_grad = True

    train_ds = ArrayTimeSeriesDataset(train_data, input_length, horizon)
    val_ds   = ArrayTimeSeriesDataset(val_data,   input_length, horizon)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=False)

    fine_tune_train_loss_arr, _, fine_tune_val_loss_arr, _ = train(
        supervised_model, train_loader, val_loader,
        criterion=criterion, epochs=10,
        optimizer=optimizer, device=device
    )
    end_val_loss.append(fine_tune_val_loss_arr[-1])

print(zip(forecast_horizon_windows, end_val_loss))

Horizon: 24
--------------------
Starting training...
Epoch 1/20


train:  37%|███▋      | 154/419 [05:16<09:05,  2.06s/it]


KeyboardInterrupt: 